# conv-channel-sum composite — cx5: channel-sum reduction produces shape predicted by conv-output-shape

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-channel-sum`, `conv-output-shape`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-channel-sum"
DD_ATOM_IDS = ["conv-channel-sum", "conv-output-shape"]
DD_SUBTOPICS = ["CNN: Channel-axis sum semantics", "CNN: Conv output shape"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The Conv2d output shape `(B, C_out, H_out, W_out)` is jointly determined by TWO atoms:

1. **conv-output-shape** — the spatial dims `H_out`, `W_out` come from the analytic formula `(H + 2*pad - K) // stride + 1`.
2. **conv-channel-sum** — the channel dim `C_out` comes from the WEIGHT, not the input — and the C_in axis disappears under summation. The output has `C_out` channels because we sum-reduce per-output-channel.

If either atom is wrong the shape is wrong:
- Forget to reduce over C_in → output has an extra C_in axis you never wanted.
- Get the conv-output-shape formula off by one → spatial dims don't match downstream.

The composition: this drill makes you write a `channel_sum_conv(x, w, stride, pad)` whose JOB is to produce the exactly-correct `(B, C_out, H_out, W_out)` tensor — with stride+pad passed parametrically, AND with the C_in reduction explicit. The test asserts both pieces independently: shape (conv-output-shape) and value (conv-channel-sum).

### Composite Exercise — channel-sum reduction produces shape predicted by conv-output-shape

**Atoms exercised together**: `conv-channel-sum`, `conv-output-shape`

Implement `cx5_channel_sum_conv(x, w, stride=1, pad=0)`.

- `x`: float tensor of shape `(B, C_in, H, W)`.
- `w`: float tensor of shape `(C_out, C_in, K, K)`.
- `stride`, `pad`: ints. Match `F.conv2d` semantics.

Return a float tensor of shape `(B, C_out, H_out, W_out)` matching `F.conv2d(x, w, stride=stride, padding=pad)`.

1. **conv-output-shape atom** — compute `H_out`, `W_out` via the formula.
2. Pad the input via `F.pad`.
3. Build a strided patch view of shape `(B, C_in, H_out, W_out, K, K)`.
4. **conv-channel-sum atom** — broadcast against `w` and `.sum` over the C_in (and kernel) axes to collapse C_in. The resulting C_out axis comes from the weight tensor.

Critical invariant the test checks: `out.shape[1] == w.shape[0]` (C_out from weight, NOT from input), `out.shape[2:] == (H_out, W_out)` (from conv-output-shape formula).

In [ ]:
def cx5_channel_sum_conv(x, w, stride=1, pad=0):
    B, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    # Atom A (conv-output-shape).
    H_out = (H + 2 * pad - KH) // stride + 1
    W_out = (W + 2 * pad - KW) // stride + 1
    # Pad + strided view.
    x_p = F.pad(x, (pad, pad, pad, pad))
    sB, sC, sH, sW = x_p.stride()
    patches = x_p.as_strided(
        size=(B, C_in, H_out, W_out, KH, KW),
        stride=(sB, sC, sH * stride, sW * stride, sH, sW),
    )
    # Atom B (conv-channel-sum): elementwise broadcast then .sum over (C_in, KH, KW).
    p = patches.unsqueeze(1)                              # (B, 1, C_in, H_out, W_out, KH, KW)
    wb = w.view(1, C_out, C_in, 1, 1, KH, KW)
    return (p * wb).sum(dim=(2, -2, -1))                  # (B, C_out, H_out, W_out)


<details><summary>Show solution — cx5</summary>

```python
def cx5_channel_sum_conv(x, w, stride=1, pad=0):
    B, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    # Atom A (conv-output-shape).
    H_out = (H + 2 * pad - KH) // stride + 1
    W_out = (W + 2 * pad - KW) // stride + 1
    # Pad + strided view.
    x_p = F.pad(x, (pad, pad, pad, pad))
    sB, sC, sH, sW = x_p.stride()
    patches = x_p.as_strided(
        size=(B, C_in, H_out, W_out, KH, KW),
        stride=(sB, sC, sH * stride, sW * stride, sH, sW),
    )
    # Atom B (conv-channel-sum): elementwise broadcast then .sum over (C_in, KH, KW).
    p = patches.unsqueeze(1)                              # (B, 1, C_in, H_out, W_out, KH, KW)
    wb = w.view(1, C_out, C_in, 1, 1, KH, KW)
    return (p * wb).sum(dim=(2, -2, -1))                  # (B, C_out, H_out, W_out)
```

The shape invariant is the key check: `out.shape[1]` comes from `w.shape[0]` (C_out, the weight's output channel count), NOT from `x.shape[1]` (C_in). That's what `conv-channel-sum` is FOR — collapsing C_in. The Case-D paired check (different C_in, same output shape) is exactly this invariant. `conv-output-shape` is what makes the spatial dims correct.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx5',
        'subtopics': ["CNN: Channel-axis sum semantics", "CNN: Conv output shape"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()